# <font color="#418FDE" size="6.5" uppercase>**Logistic Regression**</font>

>Last update: 20260708.
    
By the end of this Lecture, you will be able to:
- Explain how logistic regression converts engineering features into class probabilities. 
- Implement sigmoid prediction, binary loss intuition, and gradient updates from scratch. 
- Choose a classification threshold and evaluate resulting decisions. 


## **1. Probability Classifier**

### **1.1. Linear Score**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_05/Lecture_A/image_01_01.jpg?v=1783560382" width="250">



>* Weighted features combine into one raw score
>* Bias adjusts score before probability conversion

>* Combines feature evidence into one score
>* Learns weights to separate classes

>* Linear score is evidence, not probability
>* Positive or negative scores guide classification



In [ ]:
#@title Python Code - Linear Score

# Logistic regression starts with evidence.
# Weighted features create one score.
# Civil examples make scores concrete.

import numpy as np
import matplotlib.pyplot as plt

# Store three bridge inspection examples.
labels = ["Low concern", "Moderate concern", "High concern"]
crack_mm = np.array([1.0, 4.0, 8.0])
traffic_kvpd = np.array([12.0, 35.0, 60.0])

# Define learned feature weights.
crack_weight = 0.55
traffic_weight = 0.04
bias = -3.20

# Combine features into linear scores.
scores = bias + crack_weight * crack_mm + traffic_weight * traffic_kvpd
assert scores.shape == crack_mm.shape

# Convert scores later using sigmoid.
probabilities = 1.0 / (1.0 + np.exp(-scores))
threshold = 0.50

# Make threshold based class decisions.
decisions = probabilities >= threshold
print("Linear score = bias + weighted crack + weighted traffic")
print("Positive weights raise the evidence score.")
print("Bias is the starting evidence level.")

# Print compact example results.
for name, score, prob, decision in zip(labels, scores, probabilities, decisions):
    decision_text = "repair flag" if decision else "monitor"
    print(f"{name}: score {score:.2f}, probability {prob:.2f}, {decision_text}")

# Plot scores before probability conversion.
plt.figure(figsize=(7, 4))
plt.bar(labels, scores, color=["seagreen", "orange", "crimson"])
plt.axhline(0.0, color="black", linewidth=1)
plt.ylabel("Linear score, not probability")
plt.title("Weighted civil features become one evidence score")
plt.tight_layout()
plt.show()



### **1.2. Sigmoid Probability**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_05/Lecture_A/image_01_02.jpg?v=1783560377" width="250">



>* Sigmoid converts raw scores into probabilities
>* Probabilities make class predictions easier to interpret

>* Uncertain scores change probabilities most
>* Extreme scores flatten and stay bounded

>* Sigmoid estimates positive-class probability.
>* Probabilities guide confidence-based decisions.



In [ ]:
#@title Python Code - Sigmoid Probability

# Logistic regression maps scores into probabilities.
# Sigmoid keeps predictions between zero and one.
# Civil engineering features can support decisions.

import math
import numpy as np
import matplotlib.pyplot as plt

# Define a stable sigmoid transformation.
def sigmoid(score_value):
    clipped_value = max(min(score_value, 50), -50)
    return 1 / (1 + math.exp(-clipped_value))

# Create example inspection features.
crack_width_mm = 2.4
vibration_mm_s = 7.5
age_years = 38

# Combine features into one evidence score.
score = -4.2 + 0.9 * crack_width_mm
score = score + 0.25 * vibration_mm_s
score = score + 0.04 * age_years

# Convert the score into failure probability.
probability = sigmoid(score)
threshold = 0.50
classification = "high risk" if probability >= threshold else "low risk"

# Print a compact interpretation for learners.
print(f"Evidence score: {score:.2f}")
print(f"Sigmoid probability: {probability:.3f}")
print(f"Threshold decision: {classification}")

# Prepare scores for a sigmoid curve.
scores = np.linspace(-8, 8, 200)
probabilities = 1 / (1 + np.exp(-scores))

# Plot the probability mapping.
plt.figure(figsize=(7, 4))
plt.plot(scores, probabilities, label="sigmoid probability")
plt.scatter([score], [probability], color="red", zorder=3)
plt.axhline(threshold, color="gray", linestyle="--", label="0.50 threshold")
plt.xlabel("linear evidence score")
plt.ylabel("predicted positive-class probability")
plt.title("Sigmoid converts any score into a probability")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



### **1.3. Boundary Probability**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_05/Lecture_A/image_01_03.jpg?v=1783560380" width="250">



>* Probabilities change smoothly across the boundary
>* Boundary cases show model uncertainty

>* Scores map examples across a decision boundary
>* Boundary cases show uncertain, mixed evidence

>* Boundary probabilities guide decisions, not final labels
>* Near-boundary cases signal uncertainty and review



In [ ]:
#@title Python Code - Boundary Probability

# Boundary probability shows logistic regression uncertainty.
# Civil features become smooth class probabilities.
# The midpoint marks the uncertain decision boundary.

import math
import numpy as np
import matplotlib.pyplot as plt

# Define a stable sigmoid probability function.
def sigmoid(score):
    return 1.0 / (1.0 + math.exp(-score))

# Use one engineered civil safety feature.
crack_width_mm = np.linspace(0.0, 10.0, 101)
assert crack_width_mm.size == 101

# Center the score at the boundary crack width.
boundary_mm = 5.0
slope = 1.2
score = slope * (crack_width_mm - boundary_mm)

# Convert each score into failure probability.
probability = 1.0 / (1.0 + np.exp(-score))
threshold = 0.50
labels = probability >= threshold

# Inspect three representative bridge inspection cases.
case_widths = [3.0, 5.0, 7.0]
case_scores = [slope * (x - boundary_mm) for x in case_widths]
case_probs = [sigmoid(s) for s in case_scores]

# Print concise boundary probability interpretation.
print("Feature: crack width in millimeters")
print(f"Boundary width: {boundary_mm:.1f} mm gives probability {threshold:.2f}")
for width, prob in zip(case_widths, case_probs):
    decision = "review" if abs(prob - threshold) < 0.10 else "classify"
    print(f"{width:.1f} mm -> probability {prob:.2f} -> {decision}")

# Plot probability curve and boundary line.
plt.figure(figsize=(7, 4))
plt.plot(crack_width_mm, probability, label="Failure probability")
plt.axvline(boundary_mm, color="black", linestyle="--", label="Boundary")

# Add threshold line for balanced probability.
plt.axhline(threshold, color="gray", linestyle=":", label="0.50 probability")
plt.scatter(case_widths, case_probs, color="red", zorder=3)
plt.xlabel("Crack width (mm)")
plt.ylabel("Predicted failure probability")

# Keep the plot focused and readable.
plt.title("Boundary Probability in Logistic Regression")
plt.ylim(-0.05, 1.05)
plt.legend()
plt.grid(True, alpha=0.3)

plt.show()



## **2. Manual Model Training**

### **2.1. Sigmoid Prediction**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_05/Lecture_A/image_02_01.jpg?v=1783560373" width="250">



>* Features combine into one raw score
>* Sigmoid maps scores to class probability

>* Score features, then squash into probabilities
>* Borderline predictions change most with evidence

>* Estimate class likelihood before making labels
>* Compare probabilities to guide training corrections



In [ ]:
#@title Python Code - Sigmoid Prediction

# Manual logistic regression scoring example.
# Sigmoid converts scores into probabilities.
# Civil features influence binary decisions.

import math
import numpy as np
import matplotlib.pyplot as plt

# Store tiny bridge inspection features.
feature_names = ["crack_mm", "deflection_in", "age_years"]
features = np.array([2.0, 0.35, 42.0], dtype=float)

# Use hand chosen logistic weights.
weights = np.array([0.55, 1.80, 0.035], dtype=float)
bias = -3.20

# Validate matching feature and weight lengths.
assert features.shape == weights.shape, "Feature and weight shapes must match."
raw_score = float(np.dot(features, weights) + bias)

# Define sigmoid using the math library.
def sigmoid(score):
    return 1.0 / (1.0 + math.exp(-score))

# Convert the raw score into probability.
probability = sigmoid(raw_score)
threshold = 0.50

# Convert probability into a class decision.
decision = "repair soon" if probability >= threshold else "monitor"
print(f"Raw score: {raw_score:.3f}")
print(f"Sigmoid probability: {probability:.3f}")
print(f"Threshold decision: {decision}")

# Show sigmoid behavior across possible scores.
scores = np.linspace(-8.0, 8.0, 200)
probabilities = 1.0 / (1.0 + np.exp(-scores))

# Plot the sigmoid curve and example point.
plt.figure(figsize=(7, 4))
plt.plot(scores, probabilities, label="sigmoid(score)")
plt.scatter([raw_score], [probability], color="red", zorder=3)

# Add threshold and readable labels.
plt.axhline(threshold, color="gray", linestyle="--", label="threshold")
plt.xlabel("Raw model score")
plt.ylabel("Predicted probability")
plt.title("Sigmoid Prediction From Engineering Features")

# Keep the plot simple and focused.
plt.ylim(-0.05, 1.05)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()



### **2.2. Gradient Updates**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_05/Lecture_A/image_02_02.jpg?v=1783560371" width="250">



>* Errors guide weight adjustments after each prediction
>* Updates nudge features toward better future probabilities

>* Features influence predictions with different strengths
>* Gradients adjust weights to reduce errors

>* Update size balances learning speed and stability
>* Repeated updates refine weights and decisions



In [ ]:
#@title Python Code - Gradient Updates

# Manual logistic training uses repeated gradient updates.
# Tiny civil data keeps calculations easy.
# We learn weights without machine learning libraries.

import numpy as np
import matplotlib.pyplot as plt

# Store two engineering features per site.
X_raw = np.array([
    [2.0, 18.0], [3.0, 22.0],
    [4.0, 28.0], [5.0, 35.0],
    [6.0, 42.0], [7.0, 48.0],
    [8.0, 55.0], [9.0, 63.0]
])

# Labels mark whether repair is needed.
y = np.array([0, 0, 0, 0, 1, 1, 1, 1])

# Validate the tiny training table.
assert X_raw.shape == (8, 2)
assert y.shape == (8,)

# Standardize features for stable updates.
means = X_raw.mean(axis=0)
scales = X_raw.std(axis=0)
X = (X_raw - means) / scales

# Add an intercept column manually.
ones = np.ones((X.shape[0], 1))
X_design = np.column_stack((ones, X))

# Define the sigmoid probability function.
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

# Initialize weights and learning settings.
weights = np.zeros(X_design.shape[1])
learning_rate = 0.4
epochs = 80

# Track loss values for one plot.
loss_history = []

# Repeat prediction, error measurement, and update.
for epoch in range(epochs):
    probabilities = sigmoid(X_design @ weights)
    errors = probabilities - y
    gradient = (X_design.T @ errors) / len(y)
    weights = weights - learning_rate * gradient

    # Compute binary cross entropy safely.
    clipped = np.clip(probabilities, 1e-9, 1 - 1e-9)
    loss = -np.mean(y * np.log(clipped) + (1 - y) * np.log(1 - clipped))
    loss_history.append(loss)

# Convert final probabilities into decisions.
final_probabilities = sigmoid(X_design @ weights)
threshold = 0.50
predictions = (final_probabilities >= threshold).astype(int)

# Summarize training without printing large arrays.
accuracy = np.mean(predictions == y)
print(f"Initial loss: {loss_history[0]:.3f}")
print(f"Final loss: {loss_history[-1]:.3f}")
print(f"Learned weights: {np.round(weights, 3)}")
print(f"Threshold: {threshold:.2f}, accuracy: {accuracy:.2f}")

# Show one example probability and decision.
example_index = 4
print(f"Site {example_index} probability: {final_probabilities[example_index]:.3f}")
print(f"Site {example_index} predicted class: {predictions[example_index]}")

# Plot how gradient updates reduce loss.
plt.figure(figsize=(6, 4))
plt.plot(loss_history, color="navy", linewidth=2)
plt.xlabel("Training epoch")
plt.ylabel("Binary cross entropy loss")
plt.title("Gradient Updates Reduce Logistic Loss")
plt.grid(True, alpha=0.3)
plt.show()



### **2.3. Loss Intuition**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_05/Lecture_A/image_02_03.jpg?v=1783560375" width="250">



>* Loss measures prediction confidence against true labels
>* Wrong confident predictions trigger stronger updates

>* Loss measures surprise in predicted probabilities
>* Confident wrong predictions need stronger correction

>* Loss guides weight updates after predictions
>* Mistake severity shapes the learned boundary



In [ ]:
#@title Python Code - Loss Intuition

# Logistic loss measures prediction surprise.
# Confident wrong probabilities receive larger penalties.
# Tiny civil examples keep learning concrete.

import math
import numpy as np
import matplotlib.pyplot as plt

# Store labels and predicted failure probabilities.
labels = np.array([1, 1, 0, 0], dtype=float)
probabilities = np.array([0.90, 0.20, 0.10, 0.80], dtype=float)

# Clip probabilities to avoid logarithm problems.
epsilon = 1e-9
safe_probabilities = np.clip(probabilities, epsilon, 1 - epsilon)

# Compute binary cross entropy for each example.
losses = -(
    labels * np.log(safe_probabilities)
    + (1 - labels) * np.log(1 - safe_probabilities)
)

# Name each engineering inspection case.
cases = [
    "cracked beam, predicted high risk",
    "cracked beam, predicted low risk",
    "sound column, predicted low risk",
    "sound column, predicted high risk",
]

# Print compact loss intuition results.
print("Binary loss intuition for structural risk probabilities:")
for case, label, probability, loss in zip(cases, labels, probabilities, losses):
    print(f"label={int(label)}, p={probability:.2f}, loss={loss:.3f}: {case}")

# Summarize the average training signal.
mean_loss = float(np.mean(losses))
print(f"Average loss across four cases: {mean_loss:.3f}")

# Plot loss curves for positive and negative labels.
grid = np.linspace(0.01, 0.99, 99)
positive_loss = -np.log(grid)
negative_loss = -np.log(1 - grid)

# Create one clear teaching plot.
plt.figure(figsize=(7, 4))
plt.plot(grid, positive_loss, label="true label 1")
plt.plot(grid, negative_loss, label="true label 0")

# Mark the four example predictions.
plt.scatter(probabilities, losses, color="black", zorder=3)
plt.xlabel("Predicted probability of positive class")
plt.ylabel("Binary cross entropy loss")
plt.title("Wrong confident probabilities create large loss")
plt.legend()

# Display the single plot.
plt.tight_layout()
plt.show()



## **3. Threshold Decisions**

### **3.1. Decision Thresholds**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_05/Lecture_A/image_03_01.jpg?v=1783560385" width="250">



>* Thresholds turn probabilities into class decisions
>* Cutoffs should match real-world costs

>* Lower thresholds catch more positives
>* Higher thresholds reduce false positive actions

>* Same probabilities can justify different thresholds
>* Threshold choices affect fairness, costs, accountability



In [ ]:
#@title Python Code - Decision Thresholds

# Thresholds turn probabilities into engineering decisions.
# Different cutoffs create different mistake patterns.
# This example compares three practical thresholds.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Store predicted landslide risk probabilities.
probabilities = np.array([0.08, 0.18, 0.31, 0.44, 0.52, 0.63, 0.71, 0.86])

# Store observed outcomes from field inspection.
actual_labels = np.array([0, 0, 0, 1, 0, 1, 1, 1])

# Validate matching vector lengths before evaluation.
assert probabilities.shape == actual_labels.shape, "Probability and label lengths differ."

# Define thresholds for cautious and aggressive decisions.
thresholds = np.array([0.30, 0.50, 0.70])

# Create a compact metric table.
rows = []
for threshold in thresholds:
    predicted_labels = (probabilities >= threshold).astype(int)
    true_positive = int(((predicted_labels == 1) & (actual_labels == 1)).sum())
    false_positive = int(((predicted_labels == 1) & (actual_labels == 0)).sum())

    false_negative = int(((predicted_labels == 0) & (actual_labels == 1)).sum())
    true_negative = int(((predicted_labels == 0) & (actual_labels == 0)).sum())
    accuracy = (true_positive + true_negative) / len(actual_labels)
    rows.append([threshold, true_positive, false_positive, false_negative, accuracy])

# Convert results into a readable table.
summary = pd.DataFrame(
    rows,
    columns=["threshold", "TP", "FP", "FN", "accuracy"]
)

# Print only a small teaching summary.
print("Threshold comparison for slope-risk decisions:")
print(summary.to_string(index=False))
print("Lower thresholds catch more risky slopes but raise false alarms.")

# Plot probabilities and threshold lines.
plt.figure(figsize=(7, 4))
plt.scatter(range(1, 9), probabilities, c=actual_labels, cmap="coolwarm", s=90)

# Add threshold reference lines.
for threshold in thresholds:
    plt.axhline(threshold, linestyle="--", linewidth=1, label=f"threshold {threshold:.2f}")

# Label the decision plot clearly.
plt.xlabel("Slope site number")
plt.ylabel("Predicted landslide probability")
plt.title("Decision thresholds change classifications")
plt.legend(loc="lower right")

# Display the single teaching plot.
plt.tight_layout()
plt.show()



### **3.2. Threshold Selection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_05/Lecture_A/image_03_02.jpg?v=1783560383" width="250">



>* Set probability cutoffs for positive predictions
>* Match thresholds to costs and consequences

>* Compare false positives and false negatives
>* Balance risk, fairness, workload, and action

>* Test thresholds using validation performance metrics
>* Document and update thresholds as conditions change



In [ ]:
#@title Python Code - Threshold Selection

# Thresholds convert probabilities into engineering decisions.
# Validation data reveals practical tradeoffs clearly.
# Small examples make threshold selection understandable.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Store validation probabilities and observed outcomes.
probabilities = np.array([0.12, 0.18, 0.27, 0.33, 0.41, 0.49, 0.55, 0.62, 0.71, 0.83])
actual_damage = np.array([0, 0, 0, 1, 0, 1, 1, 0, 1, 1])

# Confirm both arrays describe identical bridge inspections.
assert probabilities.shape == actual_damage.shape
assert probabilities.size == 10

# Define candidate thresholds for maintenance alerts.
thresholds = np.array([0.30, 0.50, 0.70])
rows = []

# Evaluate each threshold using confusion counts.
for threshold in thresholds:
    predicted_alert = (probabilities >= threshold).astype(int)
    true_positive = int(((predicted_alert == 1) & (actual_damage == 1)).sum())
    false_positive = int(((predicted_alert == 1) & (actual_damage == 0)).sum())

    false_negative = int(((predicted_alert == 0) & (actual_damage == 1)).sum())
    true_negative = int(((predicted_alert == 0) & (actual_damage == 0)).sum())
    precision = true_positive / max(true_positive + false_positive, 1)
    recall = true_positive / max(true_positive + false_negative, 1)

    accuracy = (true_positive + true_negative) / probabilities.size
    rows.append([threshold, true_positive, false_positive, false_negative, accuracy, precision, recall])

# Summarize threshold tradeoffs in a compact table.
columns = ["threshold", "TP", "FP", "FN", "accuracy", "precision", "recall"]
summary = pd.DataFrame(rows, columns=columns)
print("Threshold selection for bridge damage alerts")
print(summary.round(2).to_string(index=False))

# Choose a threshold favoring fewer missed damaged bridges.
best_index = summary["recall"].idxmax()
best_threshold = summary.loc[best_index, "threshold"]
print(f"Chosen threshold for high recall: {best_threshold:.2f}")

# Plot probabilities against the chosen decision threshold.
colors = np.where(actual_damage == 1, "crimson", "steelblue")
plt.figure(figsize=(7, 4))
plt.scatter(range(1, 11), probabilities, c=colors, s=80)

plt.axhline(best_threshold, color="black", linestyle="--", label="chosen threshold")
plt.xlabel("Bridge inspection case")
plt.ylabel("Predicted damage probability")
plt.title("Threshold selection changes maintenance alerts")
plt.legend()

plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.show()



### **3.3. Decision Evaluation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_05/Lecture_A/image_03_03.jpg?v=1783560387" width="250">



>* Thresholds turn probabilities into real decisions
>* Evaluate errors by their practical consequences

>* Compare outcomes across prediction error types
>* Balance detection benefits against false alarm costs

>* Monitor thresholds as conditions change
>* Align decisions with consequences and fairness



In [ ]:
#@title Python Code - Decision Evaluation

# Evaluate threshold decisions using civil inspection probabilities.
# Compare predicted labels with true defect outcomes.
# Show how thresholds change practical mistakes.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Store small inspection examples with probabilities.
member_ids = np.arange(1, 13)
prob_defect = np.array([0.08, 0.18, 0.22, 0.31, 0.44, 0.49,
                        0.53, 0.61, 0.67, 0.74, 0.82, 0.91])
true_defect = np.array([0, 0, 1, 0, 1, 0,
                        1, 1, 0, 1, 1, 1])

# Validate matching lengths before evaluation.
assert len(member_ids) == len(prob_defect) == len(true_defect)

# Define a compact decision evaluation function.
def evaluate_threshold(threshold):
    predicted = (prob_defect >= threshold).astype(int)
    tp = int(np.sum((predicted == 1) & (true_defect == 1)))
    tn = int(np.sum((predicted == 0) & (true_defect == 0)))
    fp = int(np.sum((predicted == 1) & (true_defect == 0)))
    fn = int(np.sum((predicted == 0) & (true_defect == 1)))

    accuracy = (tp + tn) / len(true_defect)
    recall = tp / max(tp + fn, 1)
    precision = tp / max(tp + fp, 1)
    return tp, tn, fp, fn, accuracy, recall, precision

# Compare cautious and aggressive inspection thresholds.
thresholds = [0.30, 0.50, 0.70]
rows = []
for threshold in thresholds:
    values = evaluate_threshold(threshold)
    rows.append([threshold, *values])

# Build a readable summary table.
columns = ["threshold", "TP", "TN", "FP", "FN",
           "accuracy", "recall", "precision"]
summary = pd.DataFrame(rows, columns=columns)

# Print only a compact decision summary.
print("Decision evaluation for bridge member defect screening")
print(summary.round(2).to_string(index=False))
print("Lower thresholds catch more defects but create more inspections.")
print("Higher thresholds reduce false alarms but may miss defects.")

# Plot false alarms and missed defects together.
plt.figure(figsize=(7, 4))
plt.plot(summary["threshold"], summary["FP"], marker="o", label="False positives")
plt.plot(summary["threshold"], summary["FN"], marker="s", label="False negatives")
plt.xlabel("Classification threshold")
plt.ylabel("Number of cases")
plt.title("Threshold choice changes decision errors")
plt.xticks(thresholds)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Logistic Regression**</font>


In this lecture, you learned to:
- Explain how logistic regression converts engineering features into class probabilities. 
- Implement sigmoid prediction, binary loss intuition, and gradient updates from scratch. 
- Choose a classification threshold and evaluate resulting decisions. 

In the next Lecture (Lecture B), we will go over 'KNN and Trees'